# Stagnation detection in ShallowWaters.jl

## Preliminary code material

### Includes

Make sure `StagnationDetection.jl` is accessible.

In [ ]:
using Pkg

Pkg.add("ShallowWaters")
Pkg.add("Plots")
Pkg.add("Statistics")
Pkg.add("DelimitedFiles")
Pkg.add(path="path/to/stagnationdetection") # make sure StagnationDetection is visible or this will fail
# if StagnationDetection.jl

In [ ]:

using ShallowWaters
using StagnationDetection
using Plots

import DelimitedFiles: readdlm

import Statistics: mean

### Tool functions

In [ ]:
"""
Compute RMSE of `observed::Matrix` relative to `expected::Matrix`.
"""
function sim_error(observed::Matrix{T}, expected::Matrix{U}) where {T <: AbstractFloat, U <: AbstractFloat}
    @assert size(observed) == size(expected)

    sqdif = (observed .- expected) .^2

    return sqrt(sum(sqdif) / prod(size(sqdif)))
end

"""
Get the timestep `dt` from a Parameter struct used for ShallowWaters configuration
"""
function get_dt(P::Parameter)
    ShallowWaters.@unpack cfl, Lx, nx, g, H = P
    c::Float64 = √(g*H) 
    Δ::Int=Lx / nx  
    return Int(floor(cfl*Δ/c))
end

function stagnationmatrix(η::Matrix{T}) where {T <: FloatSD}
    num_adds = num_additions(first(η))
    @assert all(num_adds .== num_additions.(η))

    # a matrix of the ratio of total stagnation for each element
    stag = diff_absorptions.(η) ./ num_adds

    return stag
end

function matrixtocsv(A::Matrix, filename::String = "out.csv")
    io = open(filename, "w+")

    (M, N) = size(A)

    for row in 1:M
        for column in 1:N
            if column != N
                print(io, A[row, column], ", ")
            else
                print(io, A[row, column], "\n")
            end
        end
    end

    close(io)
end

In [ ]:
## make sure data files are accessible
# this assumes the current working directory is /shallowwaters/

cd("../data")

## Simulations

### Customiseable demo

Given a configuration, produce the appropriate shallow water simulation

In [ ]:
# generate a simulation

P = Parameter(
    T = Float16,
    Tini = Float16,
    Tcomm = Float16,
    Tprog = Float16SD,
    scale = 1,
    scale_sst = 1,
    nx = 100,
    cfl = 1.0,
)

m = run_model(P)

In [ ]:
# heatmap(Matrix(m.η'), aspectratio = 1.0)

heatmap(20:40:3.98e3, 20:40:1.98e3, Matrix(m.η'), aspect_ratio = 1.0, ylims = (0, 2e3), xlabel = "x (km)", ylabel = "y (km)")



In [ ]:
## save to file

matrixtocsv(stagnationmatrix(Matrix(m.η')), "stagtemp.dat")

matrixtocsv(Matrix(m.η'), "depthtemp.dat")

### Generate all data

Data is stored in `shallowwaters/data`, the following is provided for reproducibility.

In [ ]:
cd("data")

## float64

nx = 100
SCALES = [2.0^-i for i in 0:9]

for c in SCALES
    P = Parameter(
        T = Float64,
        nx = nx,
        scale = 1,
        scale_sst = 1,
        cfl = c
    )

    m = run_model(P)

    H = Matrix(m.η')

    depth_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "depth64.dat"

    matrixtocsv(H, depth_dname)
end


In [ ]:
## float16 no compensation

nx = 100
SCALES = [2.0^-i for i in 0:9]

for c in SCALES
    P = Parameter(
        T = Float16,
        Tini = Float16,
        Tcomm = Float16,
        Tprog = Float16SD,
        scale = 1,
        scale_sst = 1,
        nx = nx,
        cfl = c
    )

    m = run_model(P)

    H = Matrix(m.η')
    S = stagnationmatrix(H)

    depth_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "depth.dat"
    stag_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "stag.dat"

    matrixtocsv(H, depth_dname)
    matrixtocsv(S, stag_dname)
end

In [ ]:
nx = 200
SCALES = [2.0^-i for i in 0:8]

for c in SCALES
    P = Parameter(
        T = Float16,
        Tini = Float16,
        Tcomm = Float16,
        Tprog = Float16SD,
        scale = 1,
        scale_sst = 1,
        nx = nx,
        cfl = c
    )

    m = run_model(P)

    H = Matrix(m.η')
    S = stagnationmatrix(H)

    depth_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "depth.dat"
    stag_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "stag.dat"

    matrixtocsv(H, depth_dname)
    matrixtocsv(S, stag_dname)
end


In [ ]:
## float16 compensated

nx = 100
SCALES = [2.0^-i for i in 0:9]

for c in SCALES
    P = Parameter(
        T = Float16,
        scale = 1,
        scale_sst = 1,
        nx = nx,
        cfl = c,
        compensated = true
    )

    m = run_model(P)

    H = Matrix(m.η')
    # S = stagnationmatrix(H)

    depth_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "depth_comp.dat"
    # stag_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "stag_comp.dat"

    matrixtocsv(H, depth_dname)
    # matrixtocsv(S, stag_dname)
end

In [ ]:
nx = 200
SCALES = [2.0^-i for i in 0:8]

for c in SCALES
    P = Parameter(
        T = Float16,
        scale = 1,
        scale_sst = 1,
        nx = nx,
        cfl = c,
        compensated = true
    )

    m = run_model(P)

    H = Matrix(m.η')
    # S = stagnationmatrix(H)

    depth_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "depth_comp.dat"
    # stag_dname = "nx" * string(nx) * "dt" * string(get_dt(P)) * "stag_comp.dat"

    matrixtocsv(H, depth_dname)
    # matrixtocsv(S, stag_dname)
end

## Figures

### Convergence of results for smaller timestep

Measure the change in RMSE error for the ShallowWaters simulation in double precision.
Error decreases as timestep decreases.

In [ ]:
function errconv(TIMESTEP_SCALES::Vector)

    # return a sequence of timesteps not including the end value, and a corresponding error vector for adjacent values
    TIMESTEP_RANGE = zeros(lastindex(TIMESTEP_SCALES) - 1)
    E = zeros(lastindex(TIMESTEP_SCALES) - 1)

    nx = 100
    lh_dtprev = 0
    lh_dtcurr = 0

    for i in 1:lastindex(TIMESTEP_SCALES)
        cfl_current = TIMESTEP_SCALES[i]
        P = Parameter(
            T = Float64,
            scale = 1,
            scale_sst = 1,
            nx = 100,
            cfl = cfl_current
        )

        dt = get_dt(P)
        h_dname = "nx" * string(nx) * "dt" * string(dt) * "depth64.dat"

        η = readdlm(h_dname, ',', Float64)

        if i == 1
            lh_dt1 = η
            lh_dtprev = lh_dt1
            lh_dtcurr = lh_dt1

            TIMESTEP_RANGE[i] = dt
        elseif i != lastindex(TIMESTEP_SCALES)
            lh_dtcurr = η
            err = sim_error(lh_dtprev, lh_dtcurr)
            E[i-1] = err

            TIMESTEP_RANGE[i] = dt
        else
            lh_dtcurr = η
            err = sim_error(lh_dtprev, lh_dtcurr)
            E[i-1] = err
        end

        lh_dtprev = lh_dtcurr
    end

    return (TIMESTEP_RANGE, E)
end

In [ ]:
tscales = [2.0^-i for i in 0:9]

(T, E) = errconv(tscales)

In [ ]:
plot(T, E; xlabel = "Timestep dt", ylabel = "Error (RMSE) ", xscale = :log10, yscale = :log10)

### Stagnation maps

Display location of regions of stagnation in ShallowWaters Float16 computations with $nx = 100$ and $nx = 200$ grid resolutions.

In [ ]:
nx = 100
dt = 176

cd("/home/rcvw5971/Documents/Stagnation Analysis/Stagnation Detection JL/testset/shallowwaters/data/")

stag_fname = "nx" * string(nx) * "dt" * string(dt) * "stag.dat"

stag100 = readdlm(stag_fname, ',', Float64)

Lx100 = 20:40:3.98e3
Ly100 = 20:40:1.98e3

h1 = heatmap(Lx100, Ly100, stag100 .* 100, aspect_ratio = 1.0, ylims = (0, 2e3), clim = (0, 100), xlabel = "x (km)", ylabel = "y (km)")

In [ ]:
nx = 200
dt = 176

cd("/home/rcvw5971/Documents/Stagnation Analysis/Stagnation Detection JL/testset/shallowwaters/data/")

stag_fname = "nx" * string(nx) * "dt" * string(dt) * "stag.dat"

stag200 = readdlm(stag_fname, ',', Float64)

Lx200 = 10:20:3.99e3
Ly200 = 10:20:1.99e3

h2 = heatmap(Lx200, Ly200, stag200 .* 100, aspect_ratio = 1.0, ylims = (0, 2e3), clim = (0, 100), xlabel = "x (km)", ylabel = "y (km)")

### Stagnation and RMSE against `dt`

Show change of stagnation and RMSE against timestep. Compute RMSE error with regular arithmetic and compensated summation to show error changing when an alternative summation algorithm is used to mitigate stagnation. Error is measured against a reference solution.

In [ ]:
function errtrend(SCALINGS::Vector, comp::Bool = false)
    @assert issorted(SCALINGS, lt = Base.isgreater)

    minsf = minimum(SCALINGS)

    E = zeros(Float64, lastindex(SCALINGS))
    D = zeros(Int,     lastindex(SCALINGS))

    P_ref = Parameter(
        T = Float64,
        scale = 1,
        scale_sst = 1,
        nx = 100,
        cfl = minsf
    )

    dt_ref = get_dt(P_ref)

    h_ref_dname = "nx100dt" * string(dt_ref) * "depth64.dat"
    h_ref = readdlm(h_ref_dname, ',', Float64)

    for i in 1:lastindex(SCALINGS)
        P = Parameter(
            T = Float16,
            Tini = Float16,
            Tcomm = Float16,
            Tprog = Float16,
            scale = 1,
            scale_sst = 1,
            nx = 100,
            cfl = SCALINGS[i],            
            compensated = comp       
        )

        dt_current = get_dt(P)

        if !comp
            h_dname = "nx100dt" * string(dt_current) * "depth.dat"
        else
            h_dname = "nx100dt" * string(dt_current) * "depth_comp.dat"
        end

        h_current = readdlm(h_dname, ',', Float16)

        D[i] = dt_current
        E[i] = sim_error(h_current, h_ref)
    end

    return (D, E)
end

function stagbystep(SCALINGS::Vector)
    @assert issorted(SCALINGS, lt = Base.isgreater)

    S = zeros(Float64, lastindex(SCALINGS))
    D = zeros(Int,     lastindex(SCALINGS))

    for i in 1:lastindex(SCALINGS)
        P = Parameter(
            T = Float16,
            Tini = Float16,
            Tcomm = Float16,
            Tprog = Float16SD,
            scale = 1,
            scale_sst = 1,
            nx = 100,
            cfl = SCALINGS[i]
        )

        dt_current = get_dt(P)

        stag_dname = "nx100dt" * string(dt_current) * "stag.dat"
        stag = readdlm(stag_dname, ',', Float16)

        S[i] = mean(stag) * 100 # return as percentage
        D[i] = dt_current
    end

    return (D, S)
end

In [ ]:
SCLS = [2.0^-i for i in 0:9]

(D, E) = errtrend(SCLS)
(_, E_comp) = errtrend(SCLS, true)
(_, S) = stagbystep(SCLS)

In [ ]:
println("dt, error, error (comp. sum), stagnation")
for i in 1:lastindex(D)
    println(D[i], ", ", E[i], ", ", E_comp[i], ", ", S[i])
end

In [ ]:
plot(D, S, xscale = :log10, xlabel = "dt", ylabel = "Stagnation (%)", linecolor = :blue, legend = false)
plot!(twinx(), D, [E E_comp], xscale = :log10, ylabel = "RMSE", linecolor = :red, linestyle = [:solid :dash], label = ["RMSE fp16" "RMSE fp16 comp."], legend = :topright)

### Divergence

Show that the apparent result of the ShallowWaters default simulation in Float16 arithmetic diverges as the timestep `dt` decreases.

In [ ]:
Lx = 20:40:3.98e3
Ly = 20:40:1.98e3

c = 1

P = Parameter(
    T = Float16,
    scale = 1,
    scale_sst = 1,
    cfl = c
)

h = readdlm("nx100dt" * string(get_dt(P)) * "depth.dat", ',', Float16)

hmp1 = heatmap(Lx, Ly, h, aspect_ratio = 1.0, ylims = (0, 2e3), title = "cfl = " * string(c) , xlabel = "x (km)", ylabel = "y (km)")

In [ ]:
Lx = 20:40:3.98e3
Ly = 20:40:1.98e3

c = 2^-2

P = Parameter(
    T = Float16,
    # scale = 1,
    # scale_sst = 1,
    cfl = c
)

h2 = readdlm("nx100dt" * string(get_dt(P)) * "depth.dat", ',', Float16)

hmp2 = heatmap(Lx, Ly, h2, aspect_ratio = 1.0, ylims = (0, 2e3), title = "cfl = " * string(c) , xlabel = "x (km)", ylabel = "y (km)")

In [ ]:
Lx = 20:40:3.98e3
Ly = 20:40:1.98e3

c = 2^-4

P = Parameter(
    T = Float16,
    # scale = 1,
    # scale_sst = 1,
    cfl = c
)

h3 = readdlm("nx100dt" * string(get_dt(P)) * "depth.dat", ',', Float16)

hmp3 = heatmap(Lx, Ly, h3, aspect_ratio = 1.0, ylims = (0, 2e3), title = "cfl = " * string(c) , xlabel = "x (km)", ylabel = "y (km)")